In [1]:
import numpy as np
import xarray as xr

In [2]:
from data import ds_single, ds_pressure

In [3]:
import fcbench

In [4]:
def compute_clean_bitinformation(xs: np.array) -> tuple[float, ...]:
    bit_information = fcbench.metrics.BitInformation.bit_information(
        xs,
        set_zero_insignificant_confidence=0.99,
    )
    assert len(bit_information) == 32

    # find the first per-bit information maximum in the mantissa
    # round to two decimal places to improve stability
    bit_information_mantissa = bit_information[9:]
    increasing = np.round(bit_information_mantissa[1:], 2) >= np.round(
        bit_information_mantissa[:-1], 2
    )
    imax = np.where(~increasing)[0]
    if len(imax) == 0:
        # no false information at the end
        return tuple(bit_information)
    imax = imax[0]

    # find the first per-bit information maximum in the mantissa
    # round to two decimal places to improve stability
    bit_information_skyfall = bit_information[9 + imax :]
    decreasing = np.round(bit_information_skyfall[1:], 2) <= np.round(
        bit_information_skyfall[:-1], 2
    )
    imin = np.where(~decreasing)[0]
    if len(imin) == 0:
        # no false information at the end
        return tuple(bit_information)
    imin = imin[0]

    bit_information_corrected = np.copy(bit_information)
    bit_information_corrected[9 + imax + imin + 1 :] = 0.0

    return tuple(bit_information_corrected)

In [5]:
from dataclasses import dataclass


@dataclass
class BitInformationBounds:
    percentile: list[float]
    keepbits: list[int]
    linquant_absolute: list[float]
    bitround_relative: list[float]

In [6]:
def compute_bitinformation_bounds(
    da: xr.DataArray, percentile: list[float]
) -> BitInformationBounds:
    xs = da.values.flatten()

    bit_information = compute_clean_bitinformation(xs)

    if np.sum(bit_information) == 0.0:
        return BitInformationBounds(
            percentile=percentile,
            keepbits=[0.0 for _ in percentile],
            linquant_absolute=[np.inf for _ in percentile],
            bitround_relative=[np.inf for _ in percentile],
        )

    keepbitss = []
    absolutes = []
    relatives = []

    for p in percentile:
        keepbits = (
            np.searchsorted(np.cumsum(bit_information) / np.sum(bit_information), p) + 1
        )
        keepbits_mantissa = keepbits - 9

        # Without the mantissa, we can still represent every 2^k.
        # The maximum distance between consecutive 2^(k-1), 2^k, 2^(k+1) is 2^k,
        #  or 1 relative to 2^k.
        # If round-to-even is used when removing bits, the relative error will
        #  be half of that in the worst case.
        relative = np.exp2(-keepbits_mantissa) * 0.5

        # If we used linear quantisation, the absolute error would be ...
        absolute = ((np.nanmax(xs) - np.nanmin(xs)) / np.exp2(keepbits_mantissa)) * 0.5
        # If round-to-even is used, the error will again be halved

        keepbitss.append(keepbits)
        absolutes.append(absolute)
        relatives.append(relative)

    return BitInformationBounds(
        percentile=percentile,
        keepbits=keepbitss,
        linquant_absolute=absolutes,
        bitround_relative=relatives,
    )

In [7]:
from tqdm import tqdm

In [8]:
with open("bit-information-single.txt", "w") as f:
    for key in tqdm(sorted(ds_single.mean)):
        print(f"{key}: ", end="", flush=True, file=f)
        print(
            f"{compute_bitinformation_bounds(ds_single.mean[key], [1.0, 0.999, 0.99, 0.95])!r}",
            flush=True,
            file=f,
        )

100%|████████████████████████████████████████████████████████| 226/226 [15:03<00:00,  4.00s/it]


In [9]:
with open("bit-information-pressure.txt", "w") as f:
    for key in tqdm(sorted(ds_pressure.mean)):
        print(f"{key}:", flush=True, file=f)
        for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
            print(f" - {pressure} hPa: ", end="", flush=True, file=f)
            print(
                f"{compute_bitinformation_bounds(ds_pressure.mean[key].sel(pressure_level=pressure), [1.0, 0.999, 0.99, 0.95])!r}",
                flush=True,
                file=f,
            )

100%|██████████████████████████████████████████████████████████| 16/16 [04:22<00:00, 16.42s/it]
